# Synthetic packaging-machine data

Generates 547 days (about 18 months) of daily machine data and adds the derived columns Base Working Time, MME, Quality (Q), Performance (P), Availability (A) and OEE. The result is saved to `Dataset/`.

In [1]:
import numpy as np
import pandas as pd
from datetime import timedelta, datetime

In [2]:
num_days = 547
start_date = datetime(2022, 1, 1)
date_range = [start_date + timedelta(days=x) for x in range(num_days)]

In [3]:
np.random.seed(42)

data = {
    "Date": date_range,
    "Effective_Running_Time": np.round(np.random.uniform(6, 10, num_days), 1),   # hours
    "General_Running_Time": np.round(np.random.uniform(8, 12, num_days), 1),     # hours
    "Waste_Packages": np.round(np.random.uniform(50, 200, num_days)),
    "Good_Packages": np.round(np.random.uniform(800, 1000, num_days)),
    "Nominal_Speed": np.round(np.random.uniform(1000, 1200, num_days)),          # packages per hour
    "Actual_Speed": np.round(np.random.uniform(800, 1100, num_days)),            # packages per hour
    "Ancillary_Time": np.round(np.random.uniform(0.5, 1.5, num_days), 1),        # hours
    "Internal_Downtime": np.round(np.random.uniform(0.1, 0.5, num_days), 1),     # hours
    "External_Downtime": np.round(np.random.uniform(0.1, 0.3, num_days), 1),     # hours
}
df = pd.DataFrame(data)

In [4]:
# Effective time cannot exceed general time; actual speed cannot exceed nominal speed
df["Effective_Running_Time"] = df[["Effective_Running_Time", "General_Running_Time"]].min(axis=1)
df["Actual_Speed"] = df[["Actual_Speed", "Nominal_Speed"]].min(axis=1)

In [5]:
df["Base_Working_Time"] = np.round(
    df["Effective_Running_Time"] + df["Ancillary_Time"] + df["Internal_Downtime"] + df["External_Downtime"], 1)

df["MME"] = np.round(df["General_Running_Time"] / df["Effective_Running_Time"], 2)

df["Q"] = 1 - (df["Waste_Packages"] / (df["Waste_Packages"] + df["Good_Packages"]))
df["P"] = 1 - ((df["Nominal_Speed"] - df["Actual_Speed"]) / df["Nominal_Speed"])
df["A"] = 1 - ((df["Ancillary_Time"] + df["Internal_Downtime"] + df["External_Downtime"]) / df["Base_Working_Time"])
df["OEE"] = np.round(df["Q"] * df["P"] * df["A"], 2)

In [6]:
df.to_csv("Dataset/Synthetic_data_with_OEE_MME_16_06_24.csv", index=False)
df.head()

,Date,Effective_Running_Time,General_Running_Time,Waste_Packages,Good_Packages,Nominal_Speed,Actual_Speed,Ancillary_Time,Internal_Downtime,External_Downtime,Base_Working_Time,MME,Q,P,A,OEE
0,2022-01-01,7.5,10.7,85.0,968.0,1158.0,1036.0,1.4,0.2,0.2,9.3,1.43,0.919278,0.894646,0.806452,0.66
1,2022-01-02,9.8,10.1,76.0,904.0,1133.0,1097.0,1.4,0.3,0.3,11.8,1.03,0.922449,0.968226,0.830508,0.74
2,2022-01-03,8.9,11.1,83.0,925.0,1052.0,963.0,1.0,0.4,0.1,10.4,1.25,0.917659,0.915399,0.855769,0.72
3,2022-01-04,8.4,10.1,78.0,818.0,1181.0,1089.0,0.6,0.4,0.2,9.6,1.20,0.912946,0.922100,0.875000,0.74
4,2022-01-05,6.6,11.4,167.0,951.0,1134.0,823.0,0.9,0.3,0.1,7.9,1.73,0.850626,0.725750,0.835443,0.52
